# Finetuning DINOv2 with LoRA for Image Segmentation

This notebook walks through the key ideas and code behind this repository:

1. **Why DINOv2?** — self-supervised ViT features capture rich semantic structure
2. **Why LoRA?** — adapt a frozen encoder with < 1% of its parameters
3. **How the decoder works** — a single 1×1 convolution maps patch tokens → class logits
4. **PCA feature visualization** — understand what the encoder "sees" before training
5. **Training & evaluation** — run a full fine-tuning loop on Pascal VOC

## 1. Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Understanding LoRA

A standard `nn.Linear` computes $y = Wx + b$. LoRA adds a low-rank residual:

$$y = Wx + \underbrace{(B \cdot A)}_\text{rank-}r\, x \cdot \frac{\alpha}{r}$$

- $A \in \mathbb{R}^{r \times d_{in}}$  (initialized with Kaiming uniform)
- $B \in \mathbb{R}^{d_{out} \times r}$  (initialized to zero → no change at start)
- $r \ll d$, so the number of new parameters is tiny

The original weight matrix $W$ is **frozen**; only $A$ and $B$ are trained.

In [ ]:
from src.lora import LoRALinear, apply_lora, count_parameters
import torch.nn as nn

# Demonstrate LoRA on a toy linear layer
base = nn.Linear(768, 768)
lora = LoRALinear(base, rank=4, alpha=1.0)

print(lora)
x = torch.randn(2, 10, 768)
out = lora(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}')
print(f'LoRA params:  {sum(p.numel() for p in lora.parameters() if p.requires_grad):,}')
print(f'Base params:  {sum(p.numel() for p in base.parameters()):,}')

## 3. Building the Segmentation Model

```
Input image  (B, 3, H, W)
     │
     ▼
DINOv2 encoder  ── frozen weights ──┐
     │              LoRA adapters ──┘
     │  patch tokens  (B, num_patches, C)
     │  reshape → (B, C, h, w)
     ▼
1×1 conv decoder  →  (B, num_classes, h, w)
     │
     ▼
Bilinear upsample → (B, num_classes, H, W)
```

In [ ]:
from src.model import DINOSegmenter

model = DINOSegmenter(
    encoder_name='facebook/dinov2-base',
    num_classes=21,          # Pascal VOC
    patch_size=14,
    lora_rank=4,
    lora_alpha=1.0,
).to(device)

model.print_summary()

## 4. PCA Feature Visualization

Before any fine-tuning we can already see that the encoder groups semantically
similar regions.  We project the `hidden_size`-dimensional patch embeddings
down to 3 principal components and display them as an RGB image.

In [ ]:
from src.dataset import SegmentationTransform, denormalize
from src.utils import visualize_pca_features
from PIL import Image as PILImage
import numpy as np

# Load a sample image  (replace with any image path)
img_path = 'path/to/your/image.jpg'

try:
    pil_img = PILImage.open(img_path).convert('RGB')
except FileNotFoundError:
    # Generate a random image for demonstration
    pil_img = PILImage.fromarray(np.random.randint(0, 255, (448, 448, 3), dtype=np.uint8))
    print('Using random image — replace img_path with a real image.')

transform = SegmentationTransform(image_size=448, is_train=False)
dummy_mask = PILImage.fromarray(np.zeros((pil_img.height, pil_img.width), dtype=np.uint8))
tensor, _ = transform(pil_img, dummy_mask)
tensor = tensor.unsqueeze(0).to(device)

# Extract features
model.eval()
with torch.no_grad():
    features = model.get_patch_features(tensor)   # (1, C, h, w)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(denormalize(tensor[0]))
axes[0].set_title('Input image')
axes[0].axis('off')

pca_rgb = visualize_pca_features(features, n_components=3)
axes[1].imshow(pca_rgb)
axes[1].set_title('PCA of DINOv2 patch features')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### DINOv2 PCA Features

DINOv2's patch features are **less noisy** than
DINOv2's: semantically similar patches cluster more tightly, so the PCA
visualization is cleaner even without fine-tuning.

To switch to DINOv2, pass `use_dinov3=True` and set the correct hub repo:

```python
model = DINOSegmenter(
    encoder_name='dinov3_vitb14',
    use_dinov3=True,
    dinov3_hub_repo='siméoni/dinov3',  # update when officially released
    ...
)
```

## 5. Dataset — Pascal VOC 2012

In [ ]:
from src.dataset import VOCSegmentationDataset, VOC_CLASSES, VOC_COLORMAP, decode_segmap
from torch.utils.data import DataLoader

# Set download=True on first run to fetch the dataset (~2 GB)
train_ds = VOCSegmentationDataset('./data', split='train', image_size=448, download=False)
print(f'Training samples: {len(train_ds)}')

images, masks = next(iter(DataLoader(train_ds, batch_size=4, shuffle=True)))
print(f'Image batch: {images.shape}  Mask batch: {masks.shape}')
print(f'Unique class ids in batch: {masks[masks != 255].unique().tolist()}')

In [ ]:
# Visualize a few training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img_np  = denormalize(images[i])
    mask_np = masks[i].numpy()
    mask_np[mask_np == 255] = 0      # treat ignore pixels as background for display
    seg_np  = decode_segmap(mask_np, VOC_COLORMAP)

    axes[0, i].imshow(img_np);   axes[0, i].axis('off'); axes[0, i].set_title(f'Image {i}')
    axes[1, i].imshow(seg_np);   axes[1, i].axis('off'); axes[1, i].set_title('Ground truth')

plt.tight_layout()
plt.show()

## 6. Forward Pass & Loss

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(ignore_index=255)

model.train()
images_gpu = images.to(device)
masks_gpu  = masks.to(device)

logits = model(images_gpu)
loss   = criterion(logits, masks_gpu)

print(f'Logits shape : {logits.shape}')   # (B, 21, H, W)
print(f'Cross-entropy loss (untrained): {loss.item():.4f}')
print(f'  (expected ~log(21) ≈ {np.log(21):.3f} for random init)')

## 7. Training

Run from the terminal:

```bash
# Download VOC data first
python -c "from torchvision.datasets import VOCSegmentation; VOCSegmentation('./data', download=True)"

# Train (GPU strongly recommended)
python train.py --config configs/default.yaml

# Or override specific settings
python train.py --epochs 30 --lr 2e-4 --batch_size 4
```

Only **LoRA weights + decoder** are updated; the backbone stays frozen.
With rank=4 on DINOv2-Base this is ≈ 1.5 M trainable parameters out of 86 M total.

## 8. Evaluation & Prediction

In [ ]:
from src.utils import RunningMIoU

# Quick mIoU example on a single batch
metric = RunningMIoU(num_classes=21)

model.eval()
with torch.no_grad():
    logits = model(images_gpu)

metric.update(logits, masks_gpu)
results = metric.compute()
print(f"Batch mIoU (untrained): {results['miou']:.4f}")
print(f"Pixel acc  (untrained): {results['pixel_accuracy']:.4f}")

In [ ]:
# After training, load the best checkpoint and visualize predictions
# ckpt = torch.load('checkpoints/best.pth', map_location=device)
# model.load_state_dict(ckpt['model_state'], strict=False)

from src.utils import plot_predictions
model.eval()
with torch.no_grad():
    logits = model(images_gpu)

fig = plot_predictions(images, masks, logits.cpu(), VOC_COLORMAP)
plt.show()

## 9. CLI Reference

| Command | Description |
|---|---|
| `python train.py` | Train with default config |
| `python train.py --resume checkpoints/epoch_10.pth` | Resume training |
| `python evaluate.py --checkpoint checkpoints/best.pth` | Full val-set evaluation |
| `python evaluate.py ... --save_viz pred.png --save_pca pca.png` | Save visualizations |
| `python predict.py --image img.jpg --checkpoint checkpoints/best.pth --show_pca` | Single-image inference |
| `python predict.py --image folder/ ...` | Batch inference on a folder |